# 05 · The app: trigger, inject, learn

## 5.0 · Hold the key: the order of events

```
key down  ->  app, cursor text, mic, connection     (local, ~30 ms)
speaking  ->  audio streams up
key up    ->  ASR tail, cleanup, paste              (~700 ms at p99)
after     ->  watch what you fix, learn the word
```

## 5.1 · Permissions

System Settings → Privacy & Security:

- **Input Monitoring**: see keys in other apps (§5.2)
- **Accessibility**: paste, read text fields (§5.3, §5.5)

In [1]:
import os
import subprocess
import time

from AppKit import NSWorkspace
from ApplicationServices import AXIsProcessTrusted


def parent_chain(pid, depth=8):
    """Walk up the process tree. macOS grants permission to the app at the top."""
    chain = []
    for _ in range(depth):
        if pid <= 1:
            break
        ask = lambda fmt: subprocess.run(["ps", "-o", fmt, "-p", str(pid)],
                                         capture_output=True, text=True).stdout.strip()
        args, ppid = ask("args="), ask("ppid=")
        if not args or not ppid:
            break
        chain.append(args)
        pid = int(ppid)
    return chain


def app_name(argv):
    """Pull FooBar.app out of a long executable path, else just the binary name."""
    for part in argv.split("/"):
        if part.endswith(".app"):
            return part
    return argv.split()[0].split("/")[-1]


print("WHO IS ASKING  (kernel first, host app last)")
for i, argv in enumerate(parent_chain(os.getpid())):
    print(f"  {'  ' * i}\u2514\u2500 {app_name(argv)}")

print(f"\nAXIsProcessTrusted() : {AXIsProcessTrusted()}")
print("  True  -> Accessibility is granted to the host app. \u00a78.3 and \u00a78.5 will work.")
print("  False -> the toggle is OFF, or the kernel started before you flipped it.")
print("           Fix it, then RESTART THE KERNEL. macOS caches trust at process start.")

WHO IS ASKING  (kernel first, host app last)
  └─ Python.app
    └─ Cursor
      └─ Cursor.app

AXIsProcessTrusted() : True
  True  -> Accessibility is granted to the host app. §8.3 and §8.5 will work.
  False -> the toggle is OFF, or the kernel started before you flipped it.
           Fix it, then RESTART THE KERNEL. macOS caches trust at process start.


## 5.2 · Stage 1: the trigger

Hold **Right Option** (stand-in for fn). Switch to another app first. Listens for 20 s.

In [3]:
from pynput import keyboard

HOTKEY = keyboard.Key.alt_r      # right Option. see the note on fn above
LISTEN_SECONDS = 10

events, held = [], {"down": False}


def on_press(key):
    if key == HOTKEY and not held["down"]:
        held["down"] = True
        front = NSWorkspace.sharedWorkspace().frontmostApplication()   # stage 8, at the right moment
        events.append((time.perf_counter(), "DOWN", front.localizedName(), front.bundleIdentifier()))


def on_release(key):
    if key == HOTKEY and held["down"]:
        held["down"] = False
        events.append((time.perf_counter(), "UP", "", ""))


listener = keyboard.Listener(on_press=on_press, on_release=on_release)
listener.start()
print(f"listening {LISTEN_SECONDS}s.")
print("SWITCH TO ANOTHER APP NOW, then hold RIGHT OPTION a few times.")
time.sleep(LISTEN_SECONDS)
listener.stop()

if not events:
    print("\nNOTHING SEEN. The listener never got the key.")
    print("That is a permission result, not a bug: Input Monitoring is not granted to the host app.")
else:
    t0, down_t = events[0][0], None
    print(f"\n{'t':>9}  {'edge':<5} {'held':>8}  destination app")
    print("-" * 66)
    for t, edge, app, bundle in events:
        if edge == "DOWN":
            down_t = t
            print(f"{(t - t0) * 1000:8.0f}ms  {edge:<5} {'':>8}  {app}  [{bundle}]")
        else:
            print(f"{(t - t0) * 1000:8.0f}ms  {edge:<5} {(t - down_t) * 1000:7.0f}ms")
    print("-" * 66)
    print(f"presses: {sum(1 for e in events if e[1] == 'DOWN')}")
    print("the 'held' column is speech duration the user handed you for free.")

listening 10s.
SWITCH TO ANOTHER APP NOW, then hold RIGHT OPTION a few times.

        t  edge      held  destination app
------------------------------------------------------------------
       0ms  DOWN            TextEdit  [com.apple.TextEdit]
     856ms  UP        856ms
    1509ms  DOWN            TextEdit  [com.apple.TextEdit]
    2070ms  UP        561ms
------------------------------------------------------------------
presses: 2
the 'held' column is speech duration the user handed you for free.


> Empty table: grant Input Monitoring, restart the kernel.

## 5.3 · Stage 6: the paste

```
1. put our text on the clipboard
2. press ⌘V for you
```

Click into **TextEdit** first. Overwrites your clipboard.

In [4]:
import Quartz
from AppKit import NSPasteboard, NSPasteboardTypeString

KEY_V = 9              # kVK_ANSI_V, a hardware keycode. not the letter "v"
COUNTDOWN = 10

TEXT = "Injected, at stage 6. "


def inject(text):
    """Clipboard + synthetic Cmd+V. Returns per-step seconds."""
    t, m = {}, time.perf_counter
    t0 = m()

    pb = NSPasteboard.generalPasteboard()
    pb.clearContents()
    pb.setString_forType_(text, NSPasteboardTypeString)
    t["1 set clipboard"] = m() - t0

    t1 = m()
    src = Quartz.CGEventSourceCreate(Quartz.kCGEventSourceStateHIDSystemState)
    for is_down in (True, False):
        ev = Quartz.CGEventCreateKeyboardEvent(src, KEY_V, is_down)
        Quartz.CGEventSetFlags(ev, Quartz.kCGEventFlagMaskCommand)
        Quartz.CGEventPost(Quartz.kCGHIDEventTap, ev)
    t["2 post Cmd+V"] = m() - t1

    t["TOTAL"] = m() - t0
    return t


print(f"open TextEdit, click in it, then come back. pasting in {COUNTDOWN}s.")
for i in range(COUNTDOWN, 0, -1):
    print(f"  {i}...", end="\r")
    time.sleep(1)

target = NSWorkspace.sharedWorkspace().frontmostApplication()
print(f"GO -> pasting into {target.localizedName()}                 ")

timings = inject(TEXT)

print(f"\n{'step':<22}{'ms':>8}")
print("-" * 30)
for name, secs in timings.items():
    if name == "TOTAL":
        print("-" * 30)
    print(f"{name:<22}{secs * 1000:>8.2f}")

open TextEdit, click in it, then come back. pasting in 10s.
GO -> pasting into TextEdit                 

step                        ms
------------------------------
1 set clipboard           3.48
2 post Cmd+V             33.45
------------------------------
TOTAL                    36.94


Two steps, a few milliseconds.

## 5.4 · Hold a key, speak, text appears

Hold Right Option, speak, release. Text lands in the app you're in.

- Audio streams up while you talk
- Report: time after release, and after your last word

In [5]:
import asyncio
import base64
import json
import queue as _queue
from pathlib import Path

import numpy as np
import sounddevice as sd
import torch
from dotenv import load_dotenv
from elevenlabs.client import ElevenLabs
from elevenlabs.realtime.scribe import AudioFormat, CommitStrategy
from groq import Groq
from silero_vad import load_silero_vad

load_dotenv(Path("../.env"), override=True)   # override, or a rotated key never reaches a live kernel
eleven = ElevenLabs(api_key=os.environ["ELEVENLABS_API_KEY"])
groq = Groq(api_key=os.environ["GROQ_API_KEY"])
vad = load_silero_vad(onnx=True)

SAMPLE_RATE = 16_000
FRAME = 512                 # 32 ms, the only size Silero takes at 16 kHz
FRAMES_PER_SEND = 3         # ~96 ms per websocket message
VAD_THRESHOLD = 0.5
LANGUAGE = "en"
# groq deprecated llama-3.1-8b-instant on Jun 17, 2026. gpt-oss reasons and cannot be told
# not to, which costs 3x. qwen takes reasoning_effort="none" and lands on the old budget.
MODEL = "qwen/qwen3.8-27b"             
MAX_DICTATIONS = 5
MAX_HOLD_SECONDS = 60

# --- stage 8: the dictionary, on disk, so stage 7 can grow it -----------
DICT_PATH = Path("../dictionary.json")
if not DICT_PATH.exists():
    DICT_PATH.write_text(json.dumps(
        ["Wispr Flow", "Silero", "Groq", "Scribe", "Denis Ambrosov", "ONNX"], indent=1))

load_dictionary = lambda: json.loads(DICT_PATH.read_text())

STRICT = """You clean up voice dictation transcripts.

Rules:
- Output ONLY the cleaned text. No preamble, no quotes, no explanation, no sign-off.
- Remove filler words: um, uh, uhm, like, you know, I mean, so at the start of a sentence.
- Resolve false starts and self-corrections: keep only what the speaker settled on.
- Add punctuation, capitalisation and sentence breaks.
- Do NOT add any word, idea, greeting or politeness the speaker did not say.
- If the transcript ends mid-sentence, leave it mid-sentence. Do NOT complete it.
- Do NOT rephrase, summarise, reorder or change the tone.
- Keep every piece of content the speaker did say."""

PROMPT = STRICT

# --- which key means what. lives here so re-running the app cell keeps it ---
from pynput import keyboard

MODES = {keyboard.Key.alt_r: "dictate"}     # type what I said
HANDLERS = {}                               # mode -> coroutine

def cleanup(raw, dest):
    """Stage 5: clean the transcript, told which app you are in."""
    return groq.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": f"{PROMPT}\n\nCONTEXT\ndestination app: {dest}"},
                  {"role": "user", "content": f"<transcript>{raw}</transcript>"}],
        temperature=0,
        reasoning_effort="none",    # stage 5 rewrites a sentence. there is nothing to think about.
    ).choices[0].message.content.strip()


print(f"mic        : {sd.query_devices(sd.default.device[0])['name']}")
print(f"hotkey     : hold RIGHT OPTION")
print(f"model      : {MODEL}")
print(f"language   : {LANGUAGE}")
print(f"dictionary : {len(load_dictionary())} terms in {DICT_PATH}")


mic        : MacBook Air Microphone
hotkey     : hold RIGHT OPTION
model      : qwen/qwen3.8-27b
language   : en
dictionary : 5 terms in ../dictionary.json


In [10]:
# --- the two threads that feed the loop ---------------------------------
audio_q: _queue.Queue = _queue.Queue()   # PortAudio thread -> event loop
key_q: _queue.Queue = _queue.Queue()     # pynput thread    -> event loop, key-downs only
HOLD = {"key": None, "up_at": None}      # the release only needs to stamp a time
                                         # NOT `held`: 5.2's listener still owns that name
LAST = {}


def on_press(key):
    if key in MODES and HOLD["key"] is None:
        HOLD.update(key=key, up_at=None)
        front = NSWorkspace.sharedWorkspace().frontmostApplication()   # stage 8, at key-down
        key_q.put((time.perf_counter(), front.localizedName(), MODES[key]))


def on_release(key):
    if HOLD["key"] == key:
        HOLD.update(key=None, up_at=time.perf_counter())


def mic_callback(indata, frames, time_info, status):
    """PortAudio's real-time thread. One line, on purpose: block it and samples drop."""
    audio_q.put(bytes(indata))


async def record(t_down):
    """Stages 2, 3 and 4: hold the key, speak, release. Returns raw text + timings."""
    while not audio_q.empty():
        audio_q.get_nowait()
    vad.reset_states()
    state = {"partial": "", "committed": None, "t_committed": None, "server": None}

    # stage 2 first, so audio is already buffering during the handshake
    stream = sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype="int16",
                            blocksize=FRAME, callback=mic_callback)
    stream.start()

    t_ws = time.perf_counter()
    conn = await eleven.speech_to_text.realtime.connect({
        "model_id": "scribe_v2_realtime",
        "audio_format": AudioFormat.PCM_16000,
        "sample_rate": SAMPLE_RATE,
        "commit_strategy": CommitStrategy.MANUAL,      # the key owns endpointing now
        "keyterms": load_dictionary(),                 # stage 8 into stage 4
        "language_code": LANGUAGE,
    })
    ws_ms = (time.perf_counter() - t_ws) * 1000
    conn.on("partial_transcript", lambda d: state.update(partial=d.get("text", "")))
    conn.on("committed_transcript",
            lambda d: state.update(committed=d.get("text", ""), t_committed=time.perf_counter()))
    # the server says WHY before it hangs up. we were throwing that away.
    conn.on("error", lambda d: state.update(server=d))

    pending, speech_frames, total_frames = [], 0, 0
    t_last_speech = None
    try:
        while HOLD["up_at"] is None and time.perf_counter() - t_down < MAX_HOLD_SECONDS:
            try:
                chunk = audio_q.get_nowait()
            except _queue.Empty:
                await asyncio.sleep(0.004)
                continue

            frame = np.frombuffer(chunk, "<i2").astype("float32") / 32768.0
            total_frames += 1
            # stage 3: still running, but nothing waits on it any more
            if float(vad(torch.from_numpy(frame), SAMPLE_RATE).item()) > VAD_THRESHOLD:
                speech_frames += 1
                t_last_speech = time.perf_counter()    # your last word, so far

            pending.append(chunk)
            if len(pending) >= FRAMES_PER_SEND:
                try:
                    await conn.send({"audio_base_64": base64.b64encode(b"".join(pending)).decode()})
                except Exception as e:
                    state["server"] = state["server"] or {"closed_on_send": repr(e)}
                    break                        # socket is gone, stop uploading
                pending.clear()

            bar = "#" * int(min(float(np.sqrt((frame ** 2).mean())) * 300, 30))
            print(f"  recording {time.perf_counter() - t_down:4.1f}s |{bar:<30}| "
                  f"{(state['partial'] or '')[-40:]:<40}", end="\r")
    finally:
        stream.stop()
        stream.close()

    t_up = HOLD["up_at"] or time.perf_counter()
    while not audio_q.empty():                     # whatever landed after release
        pending.append(audio_q.get_nowait())
    try:
        if pending:
            await conn.send({"audio_base_64": base64.b64encode(b"".join(pending)).decode()})
        await conn.commit()                        # stage 3's old job, done by your thumb
        while state["committed"] is None and time.perf_counter() - t_up < 10:
            await asyncio.sleep(0.004)
    except Exception as e:
        state["server"] = state["server"] or {"closed_on_commit": repr(e)}
    finally:
        await conn.close()
    t_closed = time.perf_counter()                 # closing the socket is on your bill too
    print(" " * 100, end="\r")
    if state["server"] and "message_type" in state["server"]:   # our own hangup is not news
        print(f"  SERVER SAID: {state['server']}")

    return {"raw": (state["committed"] or "").strip(),
            "t_down": t_down, "t_up": t_up, "t_last": t_last_speech,
            "t_asr": state["t_committed"] or time.perf_counter(), "t_closed": t_closed,
            "ws_ms": ws_ms, "speech": speech_frames, "frames": total_frames}


def report(r, t_llm0, t_llm, inj, t_done, dest, extra=""):
    ms = lambda a, b: (b - a) * 1000
    steps = [("4 ASR after release", ms(r["t_up"], r["t_asr"]), ""),
             ("  ws close", ms(r["t_asr"], r["t_closed"]), "socket hangup before cleanup starts"),
             ("5 LLM", ms(t_llm0, t_llm), extra),
             ("6 inject", inj["TOTAL"] * 1000, "")]
    total = ms(r["t_up"], t_done)
    steps.append(("  other", total - sum(s[1] for s in steps), "python between the steps"))

    print(f"     {'held (you speaking)':<24}{ms(r['t_down'], r['t_up']):>7.0f} ms   "
          f"VAD saw speech in {r['speech']}/{r['frames']} frames")
    print(f"     {'ws handshake':<24}{r['ws_ms']:>7.0f} ms   (during speech, free)")
    for name, v, note in steps:
        print(f"     {name:<24}{v:>7.0f} ms   {note}")
    print(f"     {'RELEASE -> ON SCREEN':<24}{total:>7.0f} ms   <- Wispr p99 700")
    if r["t_last"]:
        thumb = ms(r["t_last"], r["t_up"])
        print(f"     {'last word -> release':<24}{thumb:>7.0f} ms   your thumb")
        print(f"     {'LAST WORD -> ON SCREEN':<24}{thumb + total:>7.0f} ms   <- what you feel")

In [11]:
async def dictate(t_down, dest):
    """Type what I said."""
    r = await record(t_down)
    if not r["raw"]:
        print("  (nothing transcribed. held too briefly, or no speech)")
        return

    t_llm0 = time.perf_counter()
    final = cleanup(r["raw"], dest)                # stage 5
    t_llm = time.perf_counter()

    inj = inject(final + " ")                      # stage 6
    t_done = time.perf_counter()
    LAST.update(text=final, dest=dest, at=t_done)

    print(f"  -> {dest}: {final!r}")
    print(f"     raw was : {r['raw']!r}")
    report(r, t_llm0, t_llm, inj, t_done, dest)


HANDLERS["dictate"] = dictate

listener = keyboard.Listener(on_press=on_press, on_release=on_release)
listener.start()
print(f"LIVE, {MAX_DICTATIONS} presses. armed: "
      + ", ".join(f"{k.name} -> {m}{'' if m in HANDLERS else ' (NOT ARMED, falls back to dictate)'}"
                  for k, m in MODES.items()))
print("hold the key, speak, release.\n")
try:
    for n in range(1, MAX_DICTATIONS + 1):
        while key_q.empty():
            await asyncio.sleep(0.01)
        t_down, dest, mode = key_q.get_nowait()
        print(f"[{n}] {mode} in {dest}")
        await HANDLERS.get(mode, dictate)(t_down, dest)
        print()
finally:
    listener.stop()
    print("stopped. re-run this cell for another session.")

LIVE, 5 presses. armed: alt_r -> dictate
hold the key, speak, release.

[1] dictate in TextEdit
  -> TextEdit: "Hi, Denis. Let's move the meeting on Wednesday."                                    
     raw was : "Hi, Denis. Let's move the meeting on Tuesday. Oh, no, on Wednesday."
     held (you speaking)        4724 ms   VAD saw speech in 121/144 frames
     ws handshake                367 ms   (during speech, free)
     4 ASR after release         416 ms   
       ws close                  272 ms   socket hangup before cleanup starts
     5 LLM                       270 ms   
     6 inject                      2 ms   
       other                       0 ms   python between the steps
     RELEASE -> ON SCREEN        961 ms   <- Wispr p99 700
     last word -> release         23 ms   your thumb
     LAST WORD -> ON SCREEN      984 ms   <- what you feel

stopped. re-run this cell for another session.


CancelledError: 

## 5.5 · Stage 7: the dictionary learns

In **TextEdit**, not Cursor:

1. Dictate a sentence with a name it gets wrong (§5.4)
2. Run this cell, fix the word while it counts down
3. The word lands in `dictionary.json`
4. Dictate again: now it's right

In [9]:
import difflib
import string

from ApplicationServices import (
    AXUIElementCreateSystemWide,
    AXUIElementCopyAttributeValue,
    kAXFocusedUIElementAttribute,
    kAXValueAttribute,
)

WATCH_SECONDS = 25

SYSTEM_WORDS = {w.strip().lower() for w in open("/usr/share/dict/words")}


def read_focused_text():
    """Whatever is in the focused text field. Empty on Electron, terminals, web views."""
    err, focused = AXUIElementCopyAttributeValue(
        AXUIElementCreateSystemWide(), kAXFocusedUIElementAttribute, None)
    if err != 0 or focused is None:
        return None, f"no focused element (err={err})"
    err, value = AXUIElementCopyAttributeValue(focused, kAXValueAttribute, None)
    if err != 0 or value is None:
        return None, f"element has no readable value (err={err}). Electron or a web view"
    return str(value), None


def learnable(word):
    """A correction worth remembering: not punctuation, not an ordinary English word."""
    w = word.strip(string.punctuation + string.whitespace)
    if len(w) < 3:
        return None
    if w.lower() in SYSTEM_WORDS and not w[0].isupper():
        return None                     # you fixed grammar, not vocabulary
    return w


if not LAST:
    print("nothing injected yet. run 5.4 first.")
else:
    print(f"we injected into {LAST['dest']}:\n  {LAST['text']!r}\n")
    print(f"GO AND FIX A WORD IN IT NOW. reading the field back in {WATCH_SECONDS}s.")
    for i in range(WATCH_SECONDS, 0, -1):
        print(f"  {i:3d}s ", end="\r")
        time.sleep(1)

    after, problem = read_focused_text()
    print(" " * 30, end="\r")

    if problem:
        print(f"could not read the field: {problem}")
        print("this is Electron again. try it in TextEdit, which is native Cocoa.")
    else:
        before_w, after_w = LAST["text"].split(), after.split()
        sm = difflib.SequenceMatcher(None, before_w, after_w)

        print(f"{'what changed':<40}")
        print("-" * 60)
        learned = []
        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag == "equal":
                continue
            was, now = " ".join(before_w[i1:i2]), " ".join(after_w[j1:j2])
            print(f"  {tag:<8} {was!r:<28} -> {now!r}")
            for w in after_w[j1:j2]:
                kept = learnable(w)
                if kept:
                    learned.append(kept)
                elif w.strip(string.punctuation):
                    print(f"           {'':<28}    ignored {w!r}: ordinary word")

        if not learned:
            print("\nnothing distinctive to learn. that is the filter working, not a failure.")
        else:
            dictionary = load_dictionary()
            added = [w for w in learned if w not in dictionary]
            DICT_PATH.write_text(json.dumps(dictionary + added, indent=1))
            print(f"\nLEARNED {added}")
            print(f"dictionary is now {len(dictionary + added)} terms in {DICT_PATH}")
            print("run 5.4 again and say the same sentence. stage 4 gets it right this time.")

we injected into TextEdit:
  "Hi, Dennis. Let's move the meeting on Wednesday."

GO AND FIX A WORD IN IT NOW. reading the field back in 25s.
what changed                            
------------------------------------------------------------
  replace  'Dennis.'                    -> 'Denis.'

LEARNED ['Denis']
dictionary is now 6 terms in ../dictionary.json
run 5.4 again and say the same sentence. stage 4 gets it right this time.


## The pipeline is a cycle

```
 1 trigger → 2 capture → 3 VAD → 4 ASR → 5 cleanup → 6 inject
                                   ↑                     │
                              8 context ←──── 7 learn ←──┘
```

### What is left for a real product

- menu bar app, signing, installer, updater
- Windows, iOS, Android
- reading screen text within the budget
- many languages
- **correction data from many users, and models trained on it**